In [ ]:
# Set knit directory
setwd(here::here('./02_euc_grandis'))
knitr::opts_knit$set(root.dir = setwd(here::here('./02_euc_grandis')))
getwd()

In [ ]:
options(scipen = 999)

# Save figures in specific place
knitr::opts_chunk$set(autodep        = TRUE,
                      #cache.comments = TRUE,
                      comment = "",
                      # Save figures as pdf ?
                      #dev = c( "png", "pdf"),
                  
                      # Include code?
                      echo           = TRUE,
                      error          = FALSE,
                      fig.align      = "center",
                      
                      # Path where figures are going to be store pdf single 
                      # figures
                      #fig.path       = paste0("./figures_euc_grandis", "/"),
                      fig.width      = 16,
                      fig.height     = 10,
                      message        = FALSE,
                      warning        = FALSE)

In [ ]:
library(HIEdroughtbox)
library(ggplot2)
library(scales)

# For data manipulation
library(dplyr)
library(purrr)
library(stringr)
library(tidyr)
library(lubridate)
library(here)
library(tibble)

In [ ]:
string_colors <- c(string_1 = "#eec000",
                   string_2 = "#cf544c",
                   string_3 = "#0175c3",
                   string_4 = "#878687",
                   
                   string_5 =  "#000000",
                   string_6 =  "#008b28",
                   string_7 =  "#8600b6",
                   string_8 =  "#00a0ab")

# 1. Load Droughtbox data

In [ ]:
euc_grandis_raw_data <- read_hie_droughtbox_data_folder("./raw_euc_grandis_droughtbox_data")

In [ ]:
names(euc_grandis_raw_data)

# 2. Clean Droughtbox data

Remember to remove the `eval = FALSE` from each chunk

## Plot raw droughtbox data

In [ ]:
map(euc_grandis_raw_data[!grepl("no-vpd-control", names(euc_grandis_raw_data))], 
    plot_strains_weights)

## __Data with VPD controlled__ 

### File 1: 25c

In [ ]:
euc_grandis_vpd_controlled_25c_filtered <- 
    
    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_25c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
    
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%
  
    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_date = "2026-03-30",
                           to_end_date =  "2026-03-31",
                           from_start_time = "19:00:00",
                           to_end_time = "10:50:00") %>% 
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 

    # Step done to avoid breaking the time sequence
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_vpd_controlled_25c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_vpd_controlled_25c_reshaped <- 
    reshape_droughtbox_data(euc_grandis_vpd_controlled_25c_filtered)

In [ ]:
# Check data
any(euc_grandis_vpd_controlled_25c_reshaped$temperature_measured < 25 |
    euc_grandis_vpd_controlled_25c_reshaped$temperature_measured > 25)

#### Clean reshaped data

In [ ]:
euc_grandis_vpd_controlled_25c_reshaped_cleaned <- 
  
  euc_grandis_vpd_controlled_25c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 25c
        #filter( temperature_measured <= 25)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>% 
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 10)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 10)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
        
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Visualize data before summarizing
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_vpd_controlled_25c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 2: 30c

In [ ]:
euc_grandis_vpd_controlled_30c_filtered <-

    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_30c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%
  
    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_date = "2026-03-31",
                           to_end_date =  "2026-04-01",
                           from_start_time = "21:10:00",
                           to_end_time = "09:00:00") %>% 
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 

    # Step done to avoid breaking the time sequence
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_vpd_controlled_30c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_vpd_controlled_30c_reshaped <- 
    reshape_droughtbox_data(euc_grandis_vpd_controlled_30c_filtered)

In [ ]:
# Check data
any(euc_grandis_vpd_controlled_30c_reshaped$temperature_measured < 30 |
    euc_grandis_vpd_controlled_30c_reshaped$temperature_measured > 30)

#### Clean reshaped data

In [ ]:
euc_grandis_vpd_controlled_30c_reshaped_cleaned  <- 
  
    euc_grandis_vpd_controlled_30c_reshaped %>%   
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 30c
        #filter( temperature_measured <= 30c)  %>% 
        
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 12)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 12)) %>% 
        
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
        
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        filter(!c(string_number == "string_4" & time_seconds > 36000)) %>%
      
        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Visualize data before summarizing 
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                          \(x) median(x, na.rm = TRUE))) %>% 
         
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_vpd_controlled_30c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour =factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 3: 35c

In [ ]:
euc_grandis_vpd_controlled_35c_filtered <-

    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_35c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "09:40:00",
                           to_end_time = "16:50:00") %>%
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
    
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_vpd_controlled_35c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_vpd_controlled_35c_reshaped <- 
    reshape_droughtbox_data(euc_grandis_vpd_controlled_35c_filtered)

In [ ]:
# Check data
any(euc_grandis_vpd_controlled_35c_reshaped$temperature_measured < 35 |
    euc_grandis_vpd_controlled_35c_reshaped$temperature_measured > 35)

#### Clean reshaped data

In [ ]:
euc_grandis_vpd_controlled_35c_reshaped_cleaned <- 
  
    euc_grandis_vpd_controlled_35c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 35c
        #filter( temperature_measured <= 35)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>% 
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 12)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 12)) %>% 

        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
        
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_vpd_controlled_35c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 4: 40c

In [ ]:
euc_grandis_vpd_controlled_40c_filtered <-

    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_40c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "13:00:00",
                           to_end_time = "18:30:00") %>%
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_vpd_controlled_40c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_vpd_controlled_40c_reshaped <- 
    reshape_droughtbox_data(euc_grandis_vpd_controlled_40c_filtered)

In [ ]:
# Check data
any(euc_grandis_vpd_controlled_40c_reshaped$temperature_measured < 40 |
    euc_grandis_vpd_controlled_40c_reshaped$temperature_measured >  40)

#### Clean reshaped data

In [ ]:
euc_grandis_vpd_controlled_40c_reshaped_cleaned <- 
  
    euc_grandis_vpd_controlled_40c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 40c
        #filter( temperature_measured <= 40) %>%
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 

        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 10)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 10)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>%
  
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_vpd_controlled_40c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 5: 45c

In [ ]:
euc_grandis_vpd_controlled_45c_filtered <-

    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_45c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "11:59:00",
                           to_end_time = "15:59:00") %>%
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
    
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_vpd_controlled_45c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_vpd_controlled_45c_reshaped <- 
    reshape_droughtbox_data(euc_grandis_vpd_controlled_45c_filtered)

In [ ]:
# Check data
any(euc_grandis_vpd_controlled_45c_reshaped$temperature_measured < 45 |
    euc_grandis_vpd_controlled_45c_reshaped$temperature_measured > 45)

#### Clean reshaped data

In [ ]:
euc_grandis_vpd_controlled_45c_reshaped_cleaned <- 
  
    euc_grandis_vpd_controlled_45c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 45c
        #filter( temperature_measured <= 45)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 10)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 10)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
        
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_vpd_controlled_45c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 6: 50c

In [ ]:
euc_grandis_vpd_controlled_50c_filtered <- 
    
    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_50c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "16:50:00",
                           to_end_time = "20:40:00") %>% 
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>%   
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_vpd_controlled_50c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_vpd_controlled_50c_reshaped <-
    reshape_droughtbox_data(euc_grandis_vpd_controlled_50c_filtered)

In [ ]:
# Check data
any(euc_grandis_vpd_controlled_50c_reshaped$temperature_measured < 50| 
    euc_grandis_vpd_controlled_50c_reshaped$temperature_measured > 50)

#### Clean reshaped data

In [ ]:
euc_grandis_vpd_controlled_50c_reshaped_cleaned <- 
  
    euc_grandis_vpd_controlled_50c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 50c
        #filter( temperature_measured <= 50)  %>%
        
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 12)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 12)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 12)) %>% 

        # Filter observations if required
        #filter(!c(string_number == "string_7" & string_weight_grams < 16)) %>%
        #filter(!c(string_number == "string_8" & string_weight_grams < 16))
 
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minute"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
        
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%
      
        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_vpd_controlled_50c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 7: 55c

In [ ]:
euc_grandis_vpd_controlled_55c_filtered <- 
    
    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_55c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "18:30:00",
                           to_end_time = "22:10:00") %>% 
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_vpd_controlled_55c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_vpd_controlled_55c_reshaped <-
    reshape_droughtbox_data(euc_grandis_vpd_controlled_55c_filtered)

In [ ]:
# Check data
any(euc_grandis_vpd_controlled_55c_reshaped$temperature_measured < 55| 
    euc_grandis_vpd_controlled_55c_reshaped$temperature_measured > 55)

#### Clean reshaped data

In [ ]:
euc_grandis_vpd_controlled_55c_reshaped_cleaned <- 
  
    euc_grandis_vpd_controlled_55c_reshaped %>%  
      
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 55c
        #filter(temperature_measured >= 55)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 10)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 10)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 10)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minute"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                            unit = "second")) %>% 
  
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
          
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_vpd_controlled_55c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### Merge cleaned dataframes

In [ ]:

euc_grandis_vpd_controlled_merged_cleaned_data <-
 
        list(
            euc_grandis_vpd_controlled_25c_reshaped_cleaned,
            euc_grandis_vpd_controlled_30c_reshaped_cleaned,
            euc_grandis_vpd_controlled_35c_reshaped_cleaned,
            euc_grandis_vpd_controlled_40c_reshaped_cleaned,
            euc_grandis_vpd_controlled_45c_reshaped_cleaned,
            euc_grandis_vpd_controlled_50c_reshaped_cleaned,
            euc_grandis_vpd_controlled_55c_reshaped_cleaned) %>%    
            
    reduce(rbind) %>% 
    add_column(vpd_control = factor("yes")) %>%
    arrange(string_number)

#### Plot Merged data

__Make sure that values start at 0__

In [ ]:
ggplot(data = euc_grandis_vpd_controlled_merged_cleaned_data, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +
  
    geom_point() +
    
    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors  
    scale_color_manual(values = string_colors) +
    
    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +
    
    scale_x_continuous(breaks = scales::pretty_breaks(n = 10))

In [ ]:
glimpse(euc_grandis_vpd_controlled_merged_cleaned_data)

## __Data with VPD _not_ controlled__ 

### Plot raw data

In [ ]:
map(euc_grandis_raw_data[grepl("no-vpd-control", names(euc_grandis_raw_data))], 
    plot_strains_weights)

### File 2: 30c

In [ ]:
euc_grandis_no_vpd_control_30c_filtered <-

    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_no-vpd-control_30c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%
  
    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_date = "2026-04-02",
                           to_end_date =  "2026-04-02",
                           from_start_time = "00:40:00",
                           to_end_time = "17:00:00") %>% 

    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 

    # Step done to avoid breaking the time sequence
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_no_vpd_control_30c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_no_vpd_control_30c_reshaped  <- 
    reshape_droughtbox_data(euc_grandis_no_vpd_control_30c_filtered)

In [ ]:
# Check data
any(euc_grandis_no_vpd_control_30c_reshaped$temperature_measured < 30 |
    euc_grandis_no_vpd_control_30c_reshaped$temperature_measured > 30)

#### Clean reshaped data

In [ ]:
euc_grandis_no_vpd_control_30c_reshaped_cleaned  <- 
  
    euc_grandis_no_vpd_control_30c_reshaped %>%   
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 30c
        #filter( temperature_measured <= 30c)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 
        
        ## Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 8)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 8)) %>% 
        
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
  
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Visualize data before summarizing 
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                          \(x) median(x, na.rm = TRUE))) %>% 
         
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_no_vpd_control_30c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour =factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20)) +
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 3: 35c

In [ ]:
euc_grandis_no_vpd_control_35c_filtered <-

    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_no-vpd-control_35c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_date = "2026-04-02",
                           to_end_date =  "2026-04-03",
                           from_start_time = "23:30:00",
                           to_end_time = "15:00:00") %>% 
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
    
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_no_vpd_control_35c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_no_vpd_control_35c_reshaped <- 
    reshape_droughtbox_data(euc_grandis_no_vpd_control_35c_filtered)

In [ ]:
# Check data
any(euc_grandis_vpd_controlled_35c_reshaped$temperature_measured < 35 |
    euc_grandis_vpd_controlled_35c_reshaped$temperature_measured > 35)

#### Clean reshaped data

In [ ]:
euc_grandis_no_vpd_control_35c_reshaped_cleaned <- 
  
    euc_grandis_no_vpd_control_35c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 35c
        #filter( temperature_measured <= 35)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>% 
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 8)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 8)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
        
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

  
        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_no_vpd_control_35c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 4: 40c

In [ ]:
euc_grandis_no_vpd_control_40c_filtered <-

    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_no-vpd-control_40c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "17:40:00",
                           to_end_time = "23:50:00") %>%
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_no_vpd_control_40c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_no_vpd_control_40c_reshaped <- 
    reshape_droughtbox_data(euc_grandis_no_vpd_control_40c_filtered)

In [ ]:
# Check data
any(euc_grandis_no_vpd_control_40c_reshaped$temperature_measured < 40 |
    euc_grandis_no_vpd_control_40c_reshaped$temperature_measured > 40)

#### Clean reshaped data

In [ ]:
euc_grandis_no_vpd_control_40c_reshaped_cleaned <- 
  
    euc_grandis_no_vpd_control_40c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 40c
        #filter( temperature_measured <= 40) %>%
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 

        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 8)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 8)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
  
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_no_vpd_control_40c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 5: 45c

In [ ]:
euc_grandis_no_vpd_control_45c_filtered <-

    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_no-vpd-control_45c.dat"] %>%

    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "15:50:00",
                           to_end_time = "19:00:00") %>%
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
    
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_no_vpd_control_45c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_no_vpd_control_45c_reshaped <-
    reshape_droughtbox_data(euc_grandis_no_vpd_control_45c_filtered)

In [ ]:
# Check data
any(euc_grandis_no_vpd_control_45c_reshaped$temperature_measured < 45 |
    euc_grandis_no_vpd_control_45c_reshaped$temperature_measured > 45)

#### Clean reshaped data

In [ ]:
euc_grandis_no_vpd_control_45c_reshaped_cleaned <- 
  
    euc_grandis_no_vpd_control_45c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 45c
        #filter( temperature_measured <= 45) %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 8)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 8)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 
  
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_no_vpd_control_45c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point()+

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 6: 50c

In [ ]:
euc_grandis_no_vpd_control_50c_filtered <- 
    
    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_no-vpd-control_50c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "19:40:00",
                           to_end_time = "21:50:00") %>% 
    
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>%   
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_no_vpd_control_50c_filtered, 
                     show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_no_vpd_control_50c_reshaped <-
    reshape_droughtbox_data(euc_grandis_no_vpd_control_50c_filtered)

In [ ]:
# Check data
any(euc_grandis_no_vpd_control_50c_reshaped$temperature_measured < 50| 
    euc_grandis_no_vpd_control_50c_reshaped$temperature_measured > 50)

#### Clean reshaped data

In [ ]:
euc_grandis_no_vpd_control_50c_reshaped_cleaned  <- 
  
    euc_grandis_no_vpd_control_50c_reshaped %>% 
        
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 50c
        #filter( temperature_measured <= 50)  %>%
        
        # Filter strings if required
    
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 

        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 8)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 8)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minutes"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                          unit = "second")) %>% 

        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        
        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
        
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_no_vpd_control_50c_reshaped_cleaned, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20))+
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### File 7: 55c

In [ ]:
euc_grandis_no_vpd_control_55c_filtered <- 
    
    # Select the data to be clean
    euc_grandis_raw_data["euc_grandis_apr_no-vpd-control_55c.dat"] %>% 
    
    # Transform list element to dataframe type
    as.data.frame(.) %>%

    # clean colnames
    `colnames<-`(str_extract(colnames(.), '\\b\\w+$')) %>%
  
    # Choose columns to be used
    select(tare_count_smp:strain_avg_8_microstrain_avg) %>%

    # Choose a interval of time
    filter_droughtbox_data(.,
                           from_start_time = "22:10:00",
                           to_end_time = "23:50:00") %>%
  
    # Get tare groups with more than 1 observation
    group_by(tare_count_smp) %>% 
    filter(n() > 1) %>% 
  
    # Step done to avoid breaking the time sequence 
    arrange(date_time)

#### Plot filtered data

In [ ]:
plot_strains_weights(euc_grandis_no_vpd_control_55c_filtered, show_tare_group = FALSE)

#### Reshape data

In [ ]:
euc_grandis_no_vpd_control_55c_reshaped <-
    reshape_droughtbox_data(euc_grandis_no_vpd_control_55c_filtered)

In [ ]:
# Check data
any(euc_grandis_no_vpd_control_55c_reshaped$temperature_measured < 55| 
    euc_grandis_no_vpd_control_55c_reshaped$temperature_measured > 55)

#### Clean reshaped data

In [ ]:
euc_grandis_no_vpd_control_55c_reshaped_cleaned <- 
  
    euc_grandis_no_vpd_control_55c_reshaped %>%  
      
        # Remove tare_count_smp column
        ungroup() %>%     
        select(-tare_count_smp) %>% 

        # Select Values measured at 55c
        #filter(temperature_measured >= 55)  %>% 
        
        # Filter strings if required
        #filter(!c(string_number == "string_5")) %>%
        #filter(!c(string_number == "string_6")) %>% 
    
        # Filter zeros
        filter(!c(string_number == "string_1" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_2" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_3" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_4" & string_weight_grams < 8)) %>% 
        
        filter(!c(string_number == "string_5" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_6" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_7" & string_weight_grams < 8)) %>% 
        filter(!c(string_number == "string_8" & string_weight_grams < 8)) %>% 
        
        # Average over a 5 minutes
        # Remember data is collected every 10 seconds
        mutate(date_time = floor_date(date_time, "5 minute"),
               
               time_seconds = time_length(interval(first(date_time), date_time),
                                            unit = "second")) %>% 
  
        # Filter observations
        # filter(!c(string_number == "string_3" & string_weight_grams < 18.5 & time_seconds < 4000)) %>%
        # filter(!c(string_number == "string_4" & string_weight_grams < 18.5 & time_seconds < 14000)) %>%

        group_by(string_number, time_seconds, date_time, temperature_measured) %>%     
        
        # Filter data before summarize
        summarise(across(tc_avg_deg_c_avg:string_weight_grams, 
                         \(x) median(x, na.rm = TRUE))) %>% 
          
        arrange(date_time)

In [ ]:
ggplot(data = euc_grandis_no_vpd_control_55c_reshaped_cleaned, aes(x = time_seconds,
                                                              y = string_weight_grams,
                                                              colour = factor(string_number))) +

    geom_point() +

    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors
    scale_color_manual(values = string_colors) +

    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +

    scale_y_continuous(breaks = scales::pretty_breaks(n = 20)) +
    scale_x_continuous(breaks = scales::pretty_breaks(n = 20))

### Merge cleaned dataframes

In [ ]:

euc_grandis_no_vpd_control_merged_cleaned_data <-
 
        list(
            euc_grandis_no_vpd_control_30c_reshaped_cleaned,
            euc_grandis_no_vpd_control_35c_reshaped_cleaned,
            euc_grandis_no_vpd_control_40c_reshaped_cleaned,
            euc_grandis_no_vpd_control_45c_reshaped_cleaned,
            euc_grandis_no_vpd_control_50c_reshaped_cleaned,
            euc_grandis_no_vpd_control_55c_reshaped_cleaned) %>%    
            
    reduce(rbind) %>% 
    add_column(vpd_control = factor("no")) %>%
    arrange(string_number)

#### Plot Merged data

__Make sure that values start at 0__

In [ ]:

ggplot(data = euc_grandis_no_vpd_control_merged_cleaned_data, aes(x = time_seconds,
                                                                y = string_weight_grams,
                                                                colour = factor(string_number))) +
  
    geom_point()+
    
    theme_bw() +

    # Break plot into panels
    facet_grid(~temperature_measured, scales = "free") +

    # At lines showing range between 0-3600 sec (1 hour)
    geom_vline(xintercept = 0,  linetype = "dotted") +

    # Set colors  
    scale_color_manual(values = string_colors) +
    
    # Rotate labels by 90 degrees and move legend at the bottom
    theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
          legend.position = "bottom") +
    
    scale_x_continuous(breaks = scales::pretty_breaks(n = 10))

In [ ]:
glimpse(euc_grandis_no_vpd_control_merged_cleaned_data)

## Merge datasets

In [ ]:
euc_grandis_droughtbox_data_cleaned <- 
            list(
              euc_grandis_no_vpd_control_merged_cleaned_data,
              euc_grandis_vpd_controlled_merged_cleaned_data
              ) %>%    
            
    reduce(rbind) %>%
    arrange(vpd_control, temperature_measured) %>%

    # Reorganize columns  
    select(date_time, string_number, temperature_measured, vpd_control, everything())

# 3. Add dry weights to cleaned dataframe

## Load dry weights data  

In [ ]:
euc_grandis_main_spread_sheet <- 
    
    # Read file
    readxl::read_xlsx("./euc_grandis_droughtbox_main_spreadsheet.xlsx", na = "NA") %>%
    
    # Remove columns
    select(!c(notes, number_of_parafilm_pieces, date, started_at))

In [ ]:
glimpse(euc_grandis_main_spread_sheet)

In [ ]:
euc_grandis_dry_weights <-
  
    euc_grandis_main_spread_sheet  %>% 
        
       mutate(
         leaf_dry_weight_g    = leaf_dry_weight_g, 
         sample_dry_weight_g =  ((leaf_dry_weight_g) +
                                         (stem_dry_weight_g) + 
                                         hook_weight_g + 
                                         prensa_weight_g), 
              # Add word to number 
              string_number = stringr::str_c("string_", string_number),
              .keep = "unused") %>%  
  
        select(-c(paper_bag_weight_g)) %>%
    
    na.omit()

## Merge dry weights data with droughtbox data

In [ ]:
euc_grandis_droughtbox_data_cleaned <- 

    euc_grandis_droughtbox_data_cleaned %>% 

        # Remove the word string inside the column string_number 
        mutate(set_temperature = temperature_measured, .keep = "unused") %>% 
            
        inner_join(., euc_grandis_dry_weights, 
                by = join_by(string_number, set_temperature, vpd_control)) %>% 
        
        select(-c(spcode, tree_id))

### Plot droughtbox data with dry weights

In [ ]:

#plot_no_vpd_control <- 
    euc_grandis_droughtbox_data_cleaned %>% 
        
        # Filter data
        filter(vpd_control == "no") %>% 

        ggplot(data = ., aes(x = time_seconds,
                             y = string_weight_grams,
                             colour = factor(string_number))) +
            geom_point() +
        
            theme_bw() +
            
            ggtitle("Data with no VPD control") +
        
            # Move legend at the bottom
            theme() +
        
            # Break plot into panels
            facet_grid(vars(set_temperature), vars(string_number)) +
        
            # At lines showing range between 0-3600 sec (1 hour)
            geom_vline(xintercept = 0,  linetype = "dotted") +
        
            # Set colors
            scale_color_manual(values = string_colors) +
        
            # Show dry-weights
            geom_hline(aes(yintercept = sample_dry_weight_g)) +
        
            # Rotate labels by 90 degrees
            theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
                  legend.position = "none")

In [ ]:

#plot_vpd_controlled <- 
    euc_grandis_droughtbox_data_cleaned %>% 
        
        # Filter data
        filter(vpd_control == "yes") %>% 

        ggplot(data = ., aes(x = time_seconds,
                             y = string_weight_grams,
                             colour = factor(string_number))) +
            geom_point() +
        
            theme_bw() +
            
            ggtitle("Data with VPD below 2kPa") +
        
            # Move legend at the bottom
            theme() +
        
            # Break plot into panels
            facet_grid(vars(set_temperature), vars(string_number)) +
        
            # At lines showing range between 0-3600 sec (1 hour)
            geom_vline(xintercept = 0,  linetype = "dotted") +
        
            # Set colors
            scale_color_manual(values = string_colors) +
        
            # Show dry-weights
            geom_hline(aes(yintercept = sample_dry_weight_g)) +
        
            # Rotate labels by 90 degrees
            theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
                  legend.position = "none")

# 4. Export data

In [ ]:
save(euc_grandis_droughtbox_data_cleaned, 
     file = "./euc_grandis_processed_data/euc_grandis_droughtbox_data_for_rwc.RData")